# Auditoría de Videos Publicitarios con Vertex AI y BigQuery

Este notebook guía una evaluación paso a paso para auditar videos publicitarios con Vertex AI y guardar los resultados en BigQuery.


## Paso 1) Instalación de dependencias
Ejecuta esta celda una sola vez por sesión.


In [ ]:
# Instalación de librerías necesarias
!pip install -q google-cloud-bigquery google-cloud-storage google-cloud-aiplatform ipywidgets pandas


## Paso 1.5) Autenticación con Google (recomendado)
Para este notebook **ya no se usa API key**. Vertex AI y BigQuery usan credenciales de Google (ADC), por eso se reemplaza con login de Google en Colab.


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Login de Google completado (ADC activo para Vertex AI/BigQuery)")
except Exception as exc:
    print("ℹ️ No se ejecutó autenticación automática (fuera de Colab o sesión ya autenticada):", exc)


## Paso 2) Formulario de captura de datos
- Puedes enviar **varias URLs** (una por línea).
- Las características ahora se eligen con **lista de checkboxes**.


In [ ]:
import json
import datetime
import re
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from google.cloud import aiplatform, bigquery
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
import pandas as pd

criteria_catalog = {
    "Calidad visual": [
        {"id": "imagenes_referentes", "label": "Imágenes referentes al propósito (Sí/No)"},
        {"id": "ruptura_patron", "label": "Ruptura del patrón inicial (Cumple/No cumple)"},
        {"id": "lead_in", "label": "Lead-in instantáneo (Cumple/No cumple)"},
        {"id": "velocidad_lectura", "label": "Velocidad de lectura (Alta/Media/Baja)"},
        {"id": "dinamizacion_producto", "label": "Dinamización de producto (Cumple/No cumple)"},
        {"id": "jerarquia_tamano", "label": "Jerarquía de tamaño (Cumple/No cumple)"},
        {"id": "sincronia_audio_visual", "label": "Sincronía audio-visual (Cumple/No cumple)"},
        {"id": "legibilidad_movil", "label": "Legibilidad en móvil (Cumple/No cumple)"},
        {"id": "identificacion_precoz", "label": "Identificación precoz (Cumple/No cumple)"},
        {"id": "persistencia_identidad", "label": "Persistencia de identidad (Cumple/No cumple)"},
        {"id": "cierre_marca", "label": "Cierre de marca (Cumple/No cumple)"},
        {"id": "tangibilidad_beneficio", "label": "Tangibilidad del beneficio (Cumple/No cumple)"},
        {"id": "resolucion_barreras", "label": "Resolución de barreras (Cumple/No cumple)"},
        {"id": "uso_producto", "label": "Uso de producto/servicio (Cumple/No cumple)"},
        {"id": "independencia_audio", "label": "Independencia del audio (Cumple/No cumple)"},
        {"id": "respeto_zonas", "label": "Respeto de zonas seguras (Cumple/No cumple)"},
        {"id": "contraste", "label": "Contraste fondo-figura (Cumple/No cumple)"},
    ],
    "Creatividad y branding": [
        {"id": "singularidad_mensaje", "label": "Singularidad del mensaje (Cumple/No cumple)"},
        {"id": "hook_visual", "label": "Optimización del hook visual (Alta/Media/Baja)"},
        {"id": "rebalanceo", "label": "Rebalanceo branding-producto (Alta/Media/Baja)"},
        {"id": "saturacion_textual", "label": "Gestión de saturación textual (Baja/Media/Alta)"},
    ],
}

criteria_definitions = {
    "imagenes_referentes": "Imágenes referentes al propósito (Sí/No): Las imágenes mostradas deben estar alineadas con la promoción o producto anunciado, especialmente en los primeros 3 segundos.",
    "ruptura_patron": "Ruptura del patrón inicial (Cumple/No cumple): Ocurre un cambio visual significativo (corte, animación, color) en los primeros 0-3 segundos.",
    "lead_in": "Lead-in instantáneo (Cumple/No cumple): El sonido comienza en el 00:00 sin silencios iniciales.",
    "velocidad_lectura": "Velocidad de lectura (Alta/Media/Baja): La duración de las escenas permite leer el texto completo cómodamente.",
    "dinamizacion_producto": "Dinamización de producto (Cumple/No cumple): Uso de técnicas cinematográficas y de edición para mantener la atención sostenida y evitar la fatiga visual.",
    "jerarquia_tamano": "Jerarquía de tamaño (Cumple/No cumple): El elemento más importante (%, precio, producto) es el objeto más grande en pantalla.",
    "sincronia_audio_visual": "Sincronía audio-visual (Cumple/No cumple): La locución verbal coincide con la información clave que aparece en texto.",
    "legibilidad_movil": "Legibilidad en móvil (Cumple/No cumple): Los textos principales son lo suficientemente grandes para verse en una pantalla de celular.",
    "identificacion_precoz": "Identificación precoz (Cumple/No cumple): Logo/color corporativo aparece en los primeros 5 segundos.",
    "persistencia_identidad": "Persistencia de identidad (Cumple/No cumple): Elementos de marca (logo, marco, marca de agua) visibles el 100% del tiempo.",
    "cierre_marca": "Cierre de marca (Cumple/No cumple): El video termina con una placa final de logo/slogan.",
    "tangibilidad_beneficio": "Tangibilidad del beneficio (Cumple/No cumple): El beneficio es concreto (%, $, cantidad) y no abstracto.",
    "resolucion_barreras": "Resolución de barreras (Cumple/No cumple): Los legales/condiciones son visualmente secundarios al beneficio principal.",
    "uso_producto": "Uso de producto/servicio (Cumple/No cumple): Se muestra el producto o servicio como protagonista visual.",
    "independencia_audio": "Independencia del audio (Cumple/No cumple): Se entiende la oferta y la marca si el video está en silencio.",
    "respeto_zonas": "Respeto de zonas seguras (Cumple/No cumple): Textos/logos evitan los bordes extremos (interfaz de UI).",
    "contraste": "Contraste fondo-figura (Cumple/No cumple): Existe alto contraste para facilitar la lectura rápida.",
    "singularidad_mensaje": "Singularidad del mensaje (Cumple/No cumple): Cada escena presenta una única idea principal, sin mezclar ofertas distintas.",
    "hook_visual": "Optimización del hook visual (Alta/Media/Baja): Capacidad del video para detener el scroll mediante un estímulo visual de alto impacto en los primeros 1.5 a 3 segundos.",
    "rebalanceo": "Rebalanceo branding-producto (Alta/Media/Baja): Distribución proporcional del tiempo en pantalla entre los elementos de identidad de marca y los beneficios tangibles del producto.",
    "saturacion_textual": "Gestión de saturación textual (Baja/Media/Alta): Optimización de la carga cognitiva mediante la reducción de palabras en pantalla para facilitar el procesamiento visual.",
}

compact = widgets.Layout(width='32%')
wide = widgets.Layout(width='66%')
full = widgets.Layout(width='100%')

video_urls = widgets.Textarea(description='Video URLs', placeholder='Una URL por línea', layout=widgets.Layout(width='100%', height='90px'))
video_names = widgets.Textarea(description='Nombre video', placeholder='Opcional: un nombre por línea (mismo orden que URLs)', layout=widgets.Layout(width='100%', height='90px'))

brand_name = widgets.Text(description='Brand', layout=compact)
product_name = widgets.Text(description='Producto', layout=compact)
product_category = widgets.Text(description='Categoría', layout=compact)

project_id = widgets.Text(description='Project ID', layout=compact)
project_zone = widgets.Text(description='Región', value='us-central1', layout=compact)
llm_name = widgets.Text(description='Modelo', value='gemini-1.5-flash', layout=compact)

temperature = widgets.FloatSlider(description='Temperature', value=0.2, min=0, max=1, step=0.05, layout=wide)
max_output_tokens = widgets.IntSlider(description='Max tokens', value=4096, min=512, max=8192, step=256, layout=wide)
top_p = widgets.FloatSlider(description='Top-p', value=0.95, min=0.1, max=1, step=0.05, layout=wide)
top_k = widgets.IntSlider(description='Top-k', value=40, min=0, max=100, step=5, layout=wide)

bq_dataset_name = widgets.Text(description='BQ dataset', layout=compact)
bq_table_name = widgets.Text(description='BQ table', layout=compact)

additional_context = widgets.Textarea(description='Contexto', placeholder='Notas adicionales para toda la corrida', layout=widgets.Layout(width='100%', height='80px'))

criteria_checkboxes = {}
checkbox_rows = []
for category, items in criteria_catalog.items():
    checkbox_rows.append(widgets.HTML(f"<b>{category}</b>"))
    for item in items:
        cb = widgets.Checkbox(value=True, description=item['label'], indent=False, layout=widgets.Layout(width='48%'))
        criteria_checkboxes[item['id']] = cb
        checkbox_rows.append(cb)

criteria_controls = widgets.VBox([
    widgets.HTML('<b>Características a evaluar (checkboxes)</b>'),
    widgets.HBox([
        widgets.Button(description='Seleccionar todo', button_style='info'),
        widgets.Button(description='Limpiar selección', button_style='warning')
    ]),
    widgets.VBox(checkbox_rows, layout=widgets.Layout(max_height='300px', overflow='auto'))
])

select_all_btn = criteria_controls.children[1].children[0]
clear_btn = criteria_controls.children[1].children[1]

def select_all(_):
    for cb in criteria_checkboxes.values():
        cb.value = True

def clear_all(_):
    for cb in criteria_checkboxes.values():
        cb.value = False

select_all_btn.on_click(select_all)
clear_btn.on_click(clear_all)

row_brand = widgets.HBox([brand_name, product_name, product_category], layout=full)
row_vertex = widgets.HBox([project_id, project_zone, llm_name], layout=full)
row_bq = widgets.HBox([bq_dataset_name, bq_table_name], layout=full)

display(HTML('<h3>Información de videos</h3>'))
display(video_urls)
display(video_names)
display(row_brand)
display(additional_context)

display(HTML('<h3>Configuración Vertex AI</h3>'))
display(row_vertex)
display(temperature, max_output_tokens, top_p, top_k)

display(HTML('<h3>Configuración BigQuery</h3>'))
display(row_bq)

display(criteria_controls)


## Paso 3) Prompt completo (vista previa)
Se construye dinámicamente con los criterios seleccionados.


In [ ]:
prompt_output = widgets.Textarea(layout=widgets.Layout(width='100%', height='340px'))
refresh_prompt_button = widgets.Button(description='Actualizar prompt', button_style='info')

def build_prompt(video_url_value='', video_name_value=''):
    selected_ids = [cid for cid, cb in criteria_checkboxes.items() if cb.value]
    selected_definitions = [f"- {criteria_definitions[cid]}" for cid in selected_ids]

    return f'''Actúa como un **Auditor de creatividad, efectos visuales y branding**, con especialización en neuromarketing y performance digital.

## Instrucciones
1. Analiza el video paso a paso.
2. Evalúa únicamente los criterios seleccionados usando los formatos Sí/No, Cumple/No cumple o Alta/Media/Baja.
3. Incluye resultado, justificación breve y recomendación por criterio.
4. Todos los criterios seleccionados valen lo mismo para el score final.
5. Incluye metadatos: execution_time, video_id, video_name, brand_name, product_name, product_category.

## Criterios seleccionados
{chr(10).join(selected_definitions)}

## Formato de salida (JSON estricto)
Devuelve únicamente JSON válido con este esquema:
{{
  "metadata": {{
    "execution_time": "ISO-8601",
    "video_id": "...",
    "video_name": "...",
    "brand_name": "...",
    "product_name": "...",
    "product_category": "..."
  }},
  "categories": [
    {{
      "category_name": "...",
      "criteria": [
        {{
          "criterion_id": "...",
          "criterion_name": "...",
          "result": "Sí|No|Cumple|No cumple|Alta|Media|Baja",
          "justification": "...",
          "recommendation": "..."
        }}
      ],
      "category_score": 0
    }}
  ],
  "final_score": 0,
  "global_score": 0,
  "strategic_recommendations": ["..."]
}}

## Datos de entrada
Video URL: {video_url_value}
Video name: {video_name_value}
Brand name: {brand_name.value}
Product name: {product_name.value}
Product category: {product_category.value}
Contexto adicional: {additional_context.value}
'''

def refresh_prompt(_=None):
    first_url = video_urls.value.splitlines()[0].strip() if video_urls.value.strip() else ''
    first_name = video_names.value.splitlines()[0].strip() if video_names.value.strip() else ''
    selected_count = len([1 for cb in criteria_checkboxes.values() if cb.value])
    if selected_count == 0:
        prompt_output.value = '⚠️ Selecciona al menos una característica en la lista de checkboxes.'
        return
    prompt_output.value = build_prompt(first_url, first_name)

refresh_prompt_button.on_click(refresh_prompt)
refresh_prompt()
display(refresh_prompt_button)
display(prompt_output)


## Paso 4) Envío a Vertex AI y visualización agradable
Evalúa todas las URLs cargadas (una por una) y muestra tablas por resultado.


In [ ]:
run_button = widgets.Button(description='Ejecutar evaluación masiva', button_style='success')
output_area = widgets.Output()

def parse_lines(value):
    return [line.strip() for line in value.splitlines() if line.strip()]

def extract_video_id(url):
    if 'watch?v=' in url:
        return url.split('watch?v=')[-1].split('&')[0]
    return url.rstrip('/').split('/')[-1]

def call_vertex(prompt_text, max_output_tokens_override=None):
    """Llama a Vertex AI usando Gemini (SDK actual) y fallback al API anterior."""
    token_limit = max_output_tokens_override or max_output_tokens.value
    try:
        vertexai.init(project=project_id.value, location=project_zone.value)
        model = GenerativeModel(llm_name.value)
        response = model.generate_content(
            prompt_text,
            generation_config=GenerationConfig(
                temperature=temperature.value,
                top_p=top_p.value,
                top_k=top_k.value,
                max_output_tokens=token_limit,
                response_mime_type='application/json',
            ),
        )
        if getattr(response, 'text', None):
            return response.text

        chunks = []
        for candidate in getattr(response, 'candidates', []) or []:
            content = getattr(candidate, 'content', None)
            for part in getattr(content, 'parts', []) or []:
                text = getattr(part, 'text', None)
                if text:
                    chunks.append(text)
        if chunks:
            return "\n".join(chunks)
        raise ValueError('La respuesta de Gemini no trajo texto utilizable.')

    except Exception as gemini_exc:
        try:
            from vertexai.preview.language_models import TextGenerationModel
            aiplatform.init(project=project_id.value, location=project_zone.value)
            model = TextGenerationModel.from_pretrained(llm_name.value)
            response = model.predict(
                prompt_text,
                temperature=temperature.value,
                max_output_tokens=token_limit,
                top_p=top_p.value,
                top_k=top_k.value,
            )
            return response.text
        except Exception as legacy_exc:
            raise RuntimeError(
                f"Error con SDK Gemini: {gemini_exc}. Error fallback legacy: {legacy_exc}"
            )

def sanitize_json_text(raw_text):
    text = (raw_text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\s*```$', '', text)
        text = text.strip()
    return text

def try_parse_json(raw_text):
    cleaned = sanitize_json_text(raw_text)
    try:
        return json.loads(cleaned), cleaned, None
    except json.JSONDecodeError as exc:
        return None, cleaned, exc

def looks_truncated_json(text):
    if not text:
        return False
    opens = text.count('{') + text.count('[')
    closes = text.count('}') + text.count(']')
    return opens > closes

def render_single_result(parsed, idx):
    display(Markdown(f"### Resultado {idx}"))
    meta_df = pd.DataFrame([parsed.get('metadata', {})])
    display(meta_df)
    for category in parsed.get('categories', []):
        display(Markdown(f"#### {category.get('category_name', 'Sin categoría')}"))
        cdf = pd.DataFrame(category.get('criteria', []))
        if not cdf.empty:
            display(cdf)
        display(Markdown(f"**Score categoría:** {category.get('category_score', 0)}"))
    display(Markdown(f"**Score final:** {parsed.get('final_score', 0)} | **Score global acumulado:** {parsed.get('global_score', 0)}"))

def on_run_click(_):
    with output_area:
        output_area.clear_output()
        urls = parse_lines(video_urls.value)
        names = parse_lines(video_names.value)
        if not urls:
            display(Markdown('⚠️ Debes ingresar al menos una URL.'))
            return

        selected_count = len([1 for cb in criteria_checkboxes.values() if cb.value])
        if selected_count == 0:
            display(Markdown('⚠️ Debes seleccionar al menos una característica (checkbox).'))
            return

        all_results = []
        for i, url in enumerate(urls, start=1):
            name = names[i-1] if i-1 < len(names) else f"Video {i}"
            display(Markdown(f"## Procesando video {i}/{len(urls)}"))
            prompt_text = build_prompt(url, name)

            raw_response = call_vertex(prompt_text)
            parsed, cleaned_response, parse_error = try_parse_json(raw_response)

            # Reintento automático si parece truncado o no parseable
            if parsed is None:
                is_truncated = looks_truncated_json(cleaned_response)
                retry_limit = min(8192, max(max_output_tokens.value + 1024, 4096))
                display(Markdown(
                    f"⚠️ Primer intento sin JSON válido. Reintentando con salida compacta y max_output_tokens={retry_limit}..."
                ))
                compact_prompt = prompt_text + "\n\nIMPORTANTE: Responde en JSON MINIFICADO (sin markdown, sin bloques ```), textos breves y sin relleno."
                retry_response = call_vertex(compact_prompt, max_output_tokens_override=retry_limit)
                raw_response = retry_response
                parsed, cleaned_response, parse_error = try_parse_json(raw_response)

                if parsed is None and is_truncated:
                    display(Markdown('⚠️ La respuesta parece truncada por límite de tokens. Sube Max tokens y/o reduce criterios seleccionados.'))

            if parsed is None:
                display(Markdown(f"❌ No se pudo parsear JSON para URL {url}: {parse_error}"))
                display(HTML(f"<pre>{cleaned_response[:4000]}</pre>"))

            all_results.append({
                'video_url': url,
                'video_name_input': name,
                'video_id': extract_video_id(url),
                'raw_response': raw_response,
                'parsed': parsed,
            })

            if parsed:
                render_single_result(parsed, i)

        output_area.all_results = all_results
        ok = sum(1 for r in all_results if r['parsed'] is not None)
        display(Markdown(f"### Resumen de corrida: {ok}/{len(all_results)} respuestas parseadas correctamente"))

run_button.on_click(on_run_click)
display(run_button)
display(output_area)


## Paso 5) Envío de resultados a BigQuery
Inserta todas las evaluaciones ejecutadas en el paso anterior.


In [ ]:
send_bq_button = widgets.Button(description='Enviar lote a BigQuery', button_style='primary')
bq_output = widgets.Output()

def transform_result_to_row(result):
    parsed = result.get('parsed') or {}
    metadata = parsed.get('metadata', {})
    return {
        'execution_time': metadata.get('execution_time') or datetime.datetime.now().isoformat(),
        'video_id': metadata.get('video_id') or result.get('video_id'),
        'video_name': metadata.get('video_name') or result.get('video_name_input'),
        'brand_name': metadata.get('brand_name') or brand_name.value,
        'product_name': metadata.get('product_name') or product_name.value,
        'product_category': metadata.get('product_category') or product_category.value,
        'final_score': parsed.get('final_score'),
        'global_score': parsed.get('global_score'),
        'categories_json': json.dumps(parsed.get('categories', []), ensure_ascii=False),
        'strategic_recommendations': json.dumps(parsed.get('strategic_recommendations', []), ensure_ascii=False),
        'raw_response': result.get('raw_response', ''),
        'video_url': result.get('video_url', ''),
    }

def insert_into_bigquery(rows):
    client = bigquery.Client(project=project_id.value)
    table_id = f"{project_id.value}.{bq_dataset_name.value}.{bq_table_name.value}"
    return client.insert_rows_json(table_id, rows)

def on_send_bq(_):
    with bq_output:
        bq_output.clear_output()
        results = getattr(output_area, 'all_results', [])
        if not results:
            print('Primero ejecuta el Paso 4.')
            return

        rows = [transform_result_to_row(r) for r in results]
        errors = insert_into_bigquery(rows)
        if errors == []:
            print(f"✅ {len(rows)} filas insertadas correctamente en BigQuery. Todos los pasos terminaron correctamente.")
        else:
            print('❌ Errores al insertar en BigQuery:', errors)

send_bq_button.on_click(on_send_bq)
display(send_bq_button)
display(bq_output)
